# Video Anomaly Detection — Experimental Workflow

Run this notebook from the repository root. Install dependencies and provide the benchmark datasets and YOLOv5 weights before executing the pipeline. The network checkpoints are generated by the training step.

In [ ]:
from pathlib import Path
import sys, subprocess
PROJECT_ROOT = Path.cwd()
print("Repository root:", PROJECT_ROOT)
assert (PROJECT_ROOT / "scripts" / "01_preprocess.py").exists(), "Open Jupyter from the repository root"

## Configure dataset (edit paths)
Set the dataset and detector paths below. Detector and motion-gating parameters are configurable; consult the experimental protocol for parameters not specified in the publication.

In [ ]:
DATASET="Ped1" # Ped1, Ped2, Avenue
RAW_TRAIN=r"C:/datasets/UCSDped1/Train"
RAW_TEST=r"C:/datasets/UCSDped1/Test"
GT_ROOT=r"C:/datasets/UCSDped1/Test_gt"
YOLO_REPO=r"C:/code/yolov5"
YOLO_WEIGHTS=r"C:/weights/yolov5s.pt"
PREFIX=Path("data") / DATASET.lower()
CHECKPOINT=Path("checkpoints") / DATASET.lower() / "best_model.pth"
RESULTS=Path("results") / DATASET

## 1. Preprocessing: YOLOv5 + MOG2 + ROI cropping

In [ ]:
def run(*args):
    cmd=[sys.executable,*map(str,args)]
    print(" ".join(cmd))
    subprocess.run(cmd,check=True)

run("scripts/01_preprocess.py","--input",RAW_TRAIN,"--output",PREFIX/"train","--yolov5-repo",YOLO_REPO,"--weights",YOLO_WEIGHTS,"--device","cpu")
run("scripts/01_preprocess.py","--input",RAW_TEST,"--output",PREFIX/"test","--yolov5-repo",YOLO_REPO,"--weights",YOLO_WEIGHTS,"--device","cpu")

## 2. Normal-only CNN Autoencoder training
50 epochs, batch size 128 and Adam/MSE are paper-supported. Learning rate, patience and actual augmentation are unknown.

In [ ]:
run("scripts/02_train.py","--rois",PREFIX/"train"/"rois","--output",CHECKPOINT,"--epochs","50","--batch-size","128","--device","cpu")

## 3. Validation calibration and locked test
Choose **one explicitly**. The paper is inconsistent between P95, F1-optimal and mean + k×std. Thresholds are never selected on test scores.

In [ ]:
THRESHOLD_MODE="p95"  # f1 / p95 / meanstd
run("scripts/03_evaluate.py","--dataset",DATASET,"--processed-test",PREFIX/"test","--gt-root",GT_ROOT,"--checkpoint",CHECKPOINT,"--result-dir",RESULTS,"--threshold-mode",THRESHOLD_MODE,"--device","cpu")

## 4. Inspect results and real ROC/PR images

In [ ]:
import json, numpy as np
from IPython.display import display, Image
summary=json.loads((RESULTS/"evaluation_summary.json").read_text())
print(json.dumps(summary,indent=2))
print("Saved aligned frame-level arrays:",np.load(RESULTS/"scores.npy").shape,np.load(RESULTS/"labels.npy").shape)
display(Image(filename=str(RESULTS/"roc_curve.png")))
display(Image(filename=str(RESULTS/"pr_curve.png")))